# optimal_trader MoE Paper Trading Contract Replay

This notebook is the contract-first replacement for `optimal_trader/notebooks/moe_paper_trading.ipynb`. It avoids frontend/live trading code and focuses only on historical replay artifacts.

The strategy path is:

`MoE scores or MoE feature panel -> scored_panel -> action_tape -> trade_windows -> optional options replay`

The saved `latest_scored.pkl` artifact is only one date. For a real historical backtest, set `SCORED_PANEL_PATH` or `FEATURE_PANEL_PATH` to a historical panel.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "quant_orchestrator").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PROJECT_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_orchestrator.platforms.backtesting_frameworks.optimal_trader import (
    MoePaperReplayConfig,
    load_moe_score_artifact,
    build_moe_ranked_scores,
    run_moe_paper_artifact_replay,
)
from quant_orchestrator.platforms.backtesting_frameworks.strategy_artifacts import read_strategy_artifacts

REPO_ROOT, PROJECT_ROOT


## Configuration

Leave `SCORED_PANEL_PATH` and `FEATURE_PANEL_PATH` empty for a latest-snapshot smoke run. Use one of them for an actual historical replay.

In [ ]:
SCORE_ARTIFACT = PROJECT_ROOT / "optimal_trader" / "artifacts" / "moe_paper_trading" / "latest_scored.pkl"
MODEL_ARTIFACT_DIR = PROJECT_ROOT / "optimal_trader" / "artifacts" / "synthetic_options_classifier_families"
OUTPUT_DIR = REPO_ROOT / "artifacts" / "moe_paper_contract_replay"

SCORED_PANEL_PATH = None
FEATURE_PANEL_PATH = None
BACKTEST_START = "2021-01-01"
END_DATE = "2026-06-24"
TOP_K = 40
THRESHOLD = 0.50
RUN_FMP_SYNTHETIC_OPTIONS = False
OPTION_WORKERS = 1

latest = load_moe_score_artifact(SCORE_ARTIFACT)
latest_ranked = build_moe_ranked_scores(latest, top_k=TOP_K, threshold=THRESHOLD)
display(latest_ranked.head(20))


In [ ]:
config = MoePaperReplayConfig(
    score_artifact=SCORE_ARTIFACT,
    model_artifact_dir=MODEL_ARTIFACT_DIR,
    output_dir=OUTPUT_DIR,
    feature_panel_path=FEATURE_PANEL_PATH,
    scored_panel_path=SCORED_PANEL_PATH,
    backtest_start=BACKTEST_START,
    end_date=END_DATE,
    top_k=TOP_K,
    threshold=THRESHOLD,
    run_fmp_synthetic_options=RUN_FMP_SYNTHETIC_OPTIONS,
    option_workers=OPTION_WORKERS,
)

result = run_moe_paper_artifact_replay(config)
bundle = read_strategy_artifacts(OUTPUT_DIR)

assert bundle.scored_panel is not None
assert bundle.action_tape is not None
assert bundle.trade_windows is not None
assert bundle.strategy_name == "optimal_trader.moe_paper_trading"

result.summary


In [ ]:
display(pd.DataFrame([result.summary.get("performance", {})]))
display(result.rule_replay.action_tape.head(20))
display(result.rule_replay.trade_windows.head(20))


## Contract Check

If `trade_windows` is zero, the notebook still validated the artifact contract, but the input was only the latest one-day MoE score snapshot. Supply a historical `SCORED_PANEL_PATH` or `FEATURE_PANEL_PATH` before judging strategy performance.

In [ ]:
manifest_path = OUTPUT_DIR / "strategy_artifacts_manifest.json"
{
    "manifest": str(manifest_path),
    "strategy_name": bundle.strategy_name,
    "scored_rows": 0 if bundle.scored_panel is None else len(bundle.scored_panel),
    "action_rows": 0 if bundle.action_tape is None else len(bundle.action_tape),
    "trade_windows": 0 if bundle.trade_windows is None else len(bundle.trade_windows),
}
